# Vehicle Routing Problem — Random Instance Generator & Gurobi Solver

This notebook:
1. **Generates** random VRP instances with full parameter control
2. **Visualises** the generated instance
3. **Formulates & solves** the CVRP with Gurobi (MTZ sub-tour elimination)
4. **Reports** solution metrics and plots the optimal routes

### Parameter ranges supported
| Parameter | Range / Options |
|---|---|
| Customers | 200 – 3 000 |
| Depots | 10 – 50 |
| Distance | Euclidean symmetric *or* asymmetric |
| Customer placement | Uniform, Clustered, Mixed |
| Capacity tightness | Loose → Near-infeasible |
| Demand distribution | Uniform, Normal, Exponential, Gamma |


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "gurobipy==11.0.0"])

## 0 — Imports & dependency check

In [ ]:
import os
os.environ["GRB_LICENSE_FILE"] = "gurobi.lic"

import gurobipy as gp
print(gp.gurobi.version())

import sys, subprocess

# Auto-install lightweight deps if missing (gurobipy must be installed separately)
for pkg in ["numpy", "scipy", "matplotlib", "pandas", "networkx"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
from scipy.spatial.distance import cdist
from dataclasses import dataclass, field
from typing import Literal, Optional
import math, time, warnings
warnings.filterwarnings('ignore')

# Gurobi
try:
    import gurobipy as gp
    from gurobipy import GRB
    print(f"Gurobi version: {gp.gurobi.version()}")
except ImportError:
    raise ImportError(
        "gurobipy is not installed or no licence found.\n"
        "Install with: pip install gurobipy\n"
        "Academic licence: https://www.gurobi.com/academia/academic-program-and-licenses/"
    )

print("All imports OK.")

## 1 — Configuration

In [ ]:
# ─────────────────────────────────────────────
#  EDIT THESE PARAMETERS
# ─────────────────────────────────────────────

SEED = 42  # reproducibility; None for random

# --- Scale ---
N_CUSTOMERS: int = 300          # 200 – 3 000
N_DEPOTS:    int = 5            # 10 – 50  (set lower here for quick demo)

# --- Spatial layout ---
# 'uniform'   : customers i.i.d. uniform in [0,100]²
# 'clustered' : customers grouped in K Gaussian clusters
# 'mixed'     : half uniform, half clustered
DISTRIBUTION: Literal['uniform', 'clustered', 'mixed'] = 'clustered'
N_CLUSTERS: int = 10            # only used when DISTRIBUTION in {'clustered','mixed'}
CLUSTER_STD: float = 5.0        # spread of each cluster

# --- Distance matrix ---
# 'symmetric'  : d(i,j) = d(j,i)  — Euclidean
# 'asymmetric' : perturb off-diagonal to break symmetry (large-scale realism)
DISTANCE_TYPE: Literal['symmetric', 'asymmetric'] = 'symmetric'
ASYMMETRY_FACTOR: float = 0.20  # max relative perturbation when asymmetric

# --- Demand ---
# distribution: 'uniform', 'normal', 'exponential', 'gamma'
DEMAND_DIST: Literal['uniform', 'normal', 'exponential', 'gamma'] = 'gamma'
DEMAND_MIN: float  = 1.0
DEMAND_MAX: float  = 20.0       # used as upper bound for uniform; scale for others
DEMAND_MEAN: float = 8.0        # mean for normal / exponential / gamma
DEMAND_STD:  float = 3.0        # std for normal / gamma

# --- Vehicle capacity ---
# tightness: 'loose'(0.3) | 'medium'(0.5) | 'tight'(0.75) | 'near_infeasible'(0.95)
# Capacity = total_demand / (n_vehicles * tightness_factor)
CAPACITY_TIGHTNESS: Literal['loose', 'medium', 'tight', 'near_infeasible'] = 'medium'
N_VEHICLES_PER_DEPOT: int = 3   # vehicles available per depot

# --- Solver ---
GUROBI_TIME_LIMIT: float = 300  # seconds
GUROBI_MIP_GAP:   float = 0.05  # 5 % optimality gap
FORMULATION: Literal['MTZ', 'callback'] = 'MTZ'  # 'callback' uses lazy cuts (better for large N)

# For instances with N_CUSTOMERS > 500 the MIP becomes very hard;
# the notebook will auto-switch to a cluster-and-solve heuristic.
LARGE_INSTANCE_THRESHOLD: int = 500

# ─────────────────────────────────────────────
print("Configuration loaded.")

## 2 — Instance generator

In [ ]:
@dataclass
class VRPInstance:
    """Self-contained VRP instance."""
    n_customers:  int
    n_depots:     int
    n_vehicles:   int                # total across all depots
    vehicle_cap:  float
    coords:       np.ndarray         # shape (n_depots + n_customers, 2)
    demands:      np.ndarray         # shape (n_customers,)  — depots have demand 0
    dist_matrix:  np.ndarray         # shape (n_nodes, n_nodes)
    tightness:    str
    distribution: str
    dist_type:    str
    seed:         Optional[int]

    @property
    def n_nodes(self):
        return self.n_depots + self.n_customers

    @property
    def depot_indices(self):
        return list(range(self.n_depots))

    @property
    def customer_indices(self):
        return list(range(self.n_depots, self.n_nodes))

    def summary(self):
        total_demand = self.demands.sum()
        min_vehicles_needed = math.ceil(total_demand / self.vehicle_cap)
        utilisation = total_demand / (self.n_vehicles * self.vehicle_cap)
        print(f"{'='*55}")
        print(f"  VRP Instance Summary")
        print(f"{'='*55}")
        print(f"  Customers          : {self.n_customers}")
        print(f"  Depots             : {self.n_depots}")
        print(f"  Total nodes        : {self.n_nodes}")
        print(f"  Vehicles (total)   : {self.n_vehicles}")
        print(f"  Vehicle capacity   : {self.vehicle_cap:.2f}")
        print(f"  Total demand       : {total_demand:.2f}")
        print(f"  Min vehicles needed: {min_vehicles_needed}")
        print(f"  Capacity utilisation: {utilisation:.1%}  [{self.tightness}]")
        print(f"  Spatial layout     : {self.distribution}")
        print(f"  Distance type      : {self.dist_type}")
        print(f"  Seed               : {self.seed}")
        print(f"{'='*55}")
        if self.n_vehicles < min_vehicles_needed:
            print("  ⚠️  WARNING: fleet may be too small — instance could be infeasible!")


def _generate_coords(n_customers, n_depots, distribution, n_clusters, cluster_std, rng):
    """Generate 2-D coordinates for depots + customers."""
    # Depots placed pseudo-randomly but spread across the grid
    depot_coords = rng.uniform(10, 90, size=(n_depots, 2))

    if distribution == 'uniform':
        cust_coords = rng.uniform(0, 100, size=(n_customers, 2))

    elif distribution == 'clustered':
        centres = rng.uniform(10, 90, size=(n_clusters, 2))
        assignments = rng.integers(0, n_clusters, size=n_customers)
        cust_coords = centres[assignments] + rng.normal(0, cluster_std, size=(n_customers, 2))
        cust_coords = np.clip(cust_coords, 0, 100)

    elif distribution == 'mixed':
        half = n_customers // 2
        uniform_part = rng.uniform(0, 100, size=(n_customers - half, 2))
        centres = rng.uniform(10, 90, size=(n_clusters, 2))
        assignments = rng.integers(0, n_clusters, size=half)
        cluster_part = centres[assignments] + rng.normal(0, cluster_std, size=(half, 2))
        cluster_part = np.clip(cluster_part, 0, 100)
        cust_coords = np.vstack([uniform_part, cluster_part])

    else:
        raise ValueError(f"Unknown distribution: {distribution}")

    return np.vstack([depot_coords, cust_coords])


def _generate_demands(n_customers, dist, d_min, d_max, d_mean, d_std, rng):
    """Generate customer demands."""
    if dist == 'uniform':
        demands = rng.uniform(d_min, d_max, size=n_customers)

    elif dist == 'normal':
        demands = rng.normal(d_mean, d_std, size=n_customers)
        demands = np.clip(demands, d_min, None)

    elif dist == 'exponential':
        demands = rng.exponential(d_mean, size=n_customers)
        demands = np.clip(demands, d_min, None)

    elif dist == 'gamma':
        shape = (d_mean / d_std) ** 2
        scale = d_std ** 2 / d_mean
        demands = rng.gamma(shape, scale, size=n_customers)
        demands = np.clip(demands, d_min, None)

    else:
        raise ValueError(f"Unknown demand distribution: {dist}")

    return demands.astype(float)


def _build_distance_matrix(coords, dist_type, asym_factor, rng):
    """Build (possibly asymmetric) distance matrix."""
    D = cdist(coords, coords, metric='euclidean')
    if dist_type == 'asymmetric':
        n = len(coords)
        # Add random directional perturbation — keep diagonal 0
        perturb = rng.uniform(1 - asym_factor, 1 + asym_factor, size=(n, n))
        np.fill_diagonal(perturb, 1.0)
        D = D * perturb
        np.fill_diagonal(D, 0.0)
    return D


TIGHTNESS_MAP = {
    'loose':          0.30,
    'medium':         0.50,
    'tight':          0.75,
    'near_infeasible': 0.95,
}


def generate_vrp_instance(
    n_customers=300,
    n_depots=5,
    n_vehicles_per_depot=3,
    distribution='clustered',
    n_clusters=10,
    cluster_std=5.0,
    distance_type='symmetric',
    asymmetry_factor=0.20,
    demand_dist='gamma',
    demand_min=1.0,
    demand_max=20.0,
    demand_mean=8.0,
    demand_std=3.0,
    capacity_tightness='medium',
    seed=None,
) -> VRPInstance:
    """Generate a random VRP instance."""
    # Validate
    assert 200 <= n_customers <= 3000, "n_customers must be in [200, 3000]"
    assert 1  <= n_depots     <= 50,   "n_depots must be in [1, 50]"
    assert capacity_tightness in TIGHTNESS_MAP

    rng = np.random.default_rng(seed)

    coords = _generate_coords(n_customers, n_depots, distribution, n_clusters, cluster_std, rng)
    demands = _generate_demands(n_customers, demand_dist, demand_min, demand_max,
                                demand_mean, demand_std, rng)
    dist_matrix = _build_distance_matrix(coords, distance_type, asymmetry_factor, rng)

    total_demand = demands.sum()
    n_vehicles   = n_depots * n_vehicles_per_depot
    tf           = TIGHTNESS_MAP[capacity_tightness]
    vehicle_cap  = total_demand / (n_vehicles * tf)
    # Ensure capacity >= largest single demand
    vehicle_cap  = max(vehicle_cap, demands.max())

    return VRPInstance(
        n_customers  = n_customers,
        n_depots     = n_depots,
        n_vehicles   = n_vehicles,
        vehicle_cap  = vehicle_cap,
        coords       = coords,
        demands      = demands,
        dist_matrix  = dist_matrix,
        tightness    = capacity_tightness,
        distribution = distribution,
        dist_type    = distance_type,
        seed         = seed,
    )

print("Generator defined.")

In [ ]:
# ── Generate the instance ──────────────────────────────────────────────────
instance = generate_vrp_instance(
    n_customers          = N_CUSTOMERS,
    n_depots             = N_DEPOTS,
    n_vehicles_per_depot = N_VEHICLES_PER_DEPOT,
    distribution         = DISTRIBUTION,
    n_clusters           = N_CLUSTERS,
    cluster_std          = CLUSTER_STD,
    distance_type        = DISTANCE_TYPE,
    asymmetry_factor     = ASYMMETRY_FACTOR,
    demand_dist          = DEMAND_DIST,
    demand_min           = DEMAND_MIN,
    demand_max           = DEMAND_MAX,
    demand_mean          = DEMAND_MEAN,
    demand_std           = DEMAND_STD,
    capacity_tightness   = CAPACITY_TIGHTNESS,
    seed                 = SEED,
)

instance.summary()

## 3 — Visualise the instance

In [ ]:
def plot_instance(inst: VRPInstance, ax=None, title="VRP Instance"):
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 8))

    c_idx = inst.customer_indices
    d_idx = inst.depot_indices

    sc = ax.scatter(
        inst.coords[c_idx, 0], inst.coords[c_idx, 1],
        c=inst.demands, cmap='YlOrRd', s=18, alpha=0.75,
        label='Customers', zorder=2
    )
    plt.colorbar(sc, ax=ax, label='Demand', shrink=0.7)

    ax.scatter(
        inst.coords[d_idx, 0], inst.coords[d_idx, 1],
        marker='s', s=120, c='steelblue', edgecolors='navy',
        linewidth=1.5, label='Depots', zorder=3
    )
    for i in d_idx:
        ax.annotate(f"D{i}", inst.coords[i], textcoords='offset points',
                    xytext=(4, 4), fontsize=7, color='navy')

    ax.set_title(f"{title}\n"
                 f"n={inst.n_customers} customers, {inst.n_depots} depots | "
                 f"{inst.distribution} | {inst.dist_type} dist",
                 fontsize=11)
    ax.set_xlabel("X"); ax.set_ylabel("Y")
    ax.legend(loc='lower right', fontsize=9)
    ax.set_aspect('equal')
    plt.tight_layout()
    return ax


def plot_demand_histogram(inst: VRPInstance, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(inst.demands, bins=40, color='tomato', edgecolor='white', alpha=0.85)
    ax.axvline(inst.demands.mean(), color='darkred', linestyle='--',
               label=f'Mean={inst.demands.mean():.1f}')
    ax.set_title(f"Demand distribution ({DEMAND_DIST})", fontsize=10)
    ax.set_xlabel("Demand"); ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    plt.tight_layout()
    return ax


fig, axes = plt.subplots(1, 2, figsize=(15, 7))
plot_instance(instance, ax=axes[0])
plot_demand_histogram(instance, ax=axes[1])
plt.show()

## 4 — Gurobi CVRP formulation

**Variables**
- $x_{ij}^k \in \{0,1\}$: vehicle $k$ traverses arc $(i,j)$
- $u_i^k \geq 0$: MTZ position variable (sub-tour elimination)

**Objective** — minimise total travel distance

**Constraints**
1. Every customer visited exactly once
2. Flow conservation at each node
3. Vehicles depart from and return to their home depot
4. Capacity: total demand on each route $\leq Q$
5. MTZ sub-tour elimination

In [ ]:
def solve_vrp_mtz(inst: VRPInstance,
                  time_limit: float = 300,
                  mip_gap: float = 0.05,
                  verbose: bool = True):
    """
    Solve CVRP with Gurobi using MTZ sub-tour elimination.
    Multi-depot: each depot has its own pool of vehicles;
    each vehicle departs and returns to its home depot.
    """
    D   = inst.dist_matrix
    Q   = inst.vehicle_cap
    C   = inst.customer_indices          # customer node ids
    dep = inst.depot_indices             # depot node ids
    K   = inst.n_vehicles                # total vehicle count
    vpd = inst.n_vehicles // inst.n_depots  # vehicles per depot
    N   = inst.n_nodes

    # Map each vehicle k → its home depot index
    vehicle_depot = {k: dep[k // vpd] for k in range(K)}

    nodes = list(range(N))
    arcs  = [(i, j) for i in nodes for j in nodes if i != j]

    t0 = time.time()
    env = gp.Env(empty=True)
    env.setParam('OutputFlag', int(verbose))
    env.start()
    m = gp.Model("CVRP_MTZ", env=env)
    m.Params.TimeLimit   = time_limit
    m.Params.MIPGap      = mip_gap
    m.Params.Threads     = 0          # use all available cores

    # ── Variables ──────────────────────────────────────────────────────────
    x = m.addVars(K, nodes, nodes, vtype=GRB.BINARY, name="x")
    u = m.addVars(K, C,            vtype=GRB.CONTINUOUS, lb=0, ub=Q, name="u")

    # ── Objective ──────────────────────────────────────────────────────────
    m.setObjective(
        gp.quicksum(D[i][j] * x[k, i, j]
                    for k in range(K) for (i, j) in arcs),
        GRB.MINIMIZE
    )

    # ── C1: each customer visited exactly once ──────────────────────────────
    for j in C:
        m.addConstr(
            gp.quicksum(x[k, i, j] for k in range(K) for i in nodes if i != j) == 1,
            name=f"visit_{j}"
        )

    # ── C2: flow conservation ───────────────────────────────────────────────
    for k in range(K):
        for h in C:
            m.addConstr(
                gp.quicksum(x[k, i, h] for i in nodes if i != h) ==
                gp.quicksum(x[k, h, j] for j in nodes if j != h),
                name=f"flow_{k}_{h}"
            )

    # ── C3: each vehicle departs/returns to its home depot ──────────────────
    for k in range(K):
        d_k = vehicle_depot[k]
        m.addConstr(
            gp.quicksum(x[k, d_k, j] for j in nodes if j != d_k) <= 1,
            name=f"depart_{k}"
        )
        m.addConstr(
            gp.quicksum(x[k, i, d_k] for i in nodes if i != d_k) <= 1,
            name=f"return_{k}"
        )
        # No cross-depot arcs
        for d_other in dep:
            if d_other != d_k:
                for j in nodes:
                    if j != d_other:
                        m.addConstr(x[k, d_other, j] == 0)
                        m.addConstr(x[k, j, d_other] == 0)

    # ── C4 & C5: capacity + MTZ sub-tour elimination ────────────────────────
    demands_all = np.zeros(N)
    for idx, c in enumerate(C):
        demands_all[c] = inst.demands[idx]

    for k in range(K):
        for i in C:
            for j in C:
                if i != j:
                    m.addConstr(
                        u[k, j] >= u[k, i] + demands_all[j] * x[k, i, j]
                        - Q * (1 - x[k, i, j]),
                        name=f"mtz_{k}_{i}_{j}"
                    )
        for d_k in [vehicle_depot[k]]:
            for j in C:
                m.addConstr(
                    u[k, j] >= demands_all[j] * x[k, d_k, j],
                    name=f"mtz_start_{k}_{j}"
                )

    print(f"\nModel built in {time.time()-t0:.1f}s  "
          f"| Variables: {m.NumVars:,}  Constraints: {m.NumConstrs:,}")

    m.optimize()

    solve_time = time.time() - t0
    status = m.Status

    result = {
        'status':     m.Status,
        'obj':        m.ObjVal   if status in (GRB.OPTIMAL, GRB.TIME_LIMIT) else None,
        'gap':        m.MIPGap   if status in (GRB.OPTIMAL, GRB.TIME_LIMIT) else None,
        'solve_time': solve_time,
        'routes':     {},
        'model':      m,
    }

    if status in (GRB.OPTIMAL, GRB.TIME_LIMIT):
        # Extract routes
        for k in range(K):
            d_k    = vehicle_depot[k]
            active = {(i, j) for (i, j) in arcs
                      if x[k, i, j].X > 0.5}
            if not active:
                continue
            # Reconstruct route starting from depot
            route = [d_k]
            current = d_k
            for _ in range(N):
                nxt = next((j for (i, j) in active if i == current), None)
                if nxt is None or nxt == d_k:
                    route.append(d_k)
                    break
                route.append(nxt)
                active.discard((current, nxt))
                current = nxt
            result['routes'][k] = route

    return result


print("Gurobi MTZ solver defined.")

In [ ]:
# Callback-based formulation (better for larger instances)
def solve_vrp_callback(inst: VRPInstance,
                       time_limit: float = 300,
                       mip_gap: float = 0.05,
                       verbose: bool = True):
    """
    CVRP with Gurobi using lazy sub-tour elimination via callback.
    More scalable than MTZ for large instances.
    """
    D   = inst.dist_matrix
    Q   = inst.vehicle_cap
    C   = inst.customer_indices
    dep = inst.depot_indices
    K   = inst.n_vehicles
    vpd = max(1, inst.n_vehicles // inst.n_depots)
    N   = inst.n_nodes
    nodes = list(range(N))
    arcs  = [(i, j) for i in nodes for j in nodes if i != j]
    vehicle_depot = {k: dep[k // vpd] for k in range(K)}

    demands_all = np.zeros(N)
    for idx, c in enumerate(C):
        demands_all[c] = inst.demands[idx]

    t0 = time.time()
    env = gp.Env(empty=True)
    env.setParam('OutputFlag', int(verbose))
    env.start()
    m = gp.Model("CVRP_CB", env=env)
    m.Params.TimeLimit     = time_limit
    m.Params.MIPGap        = mip_gap
    m.Params.LazyConstraints = 1
    m.Params.Threads       = 0

    x = m.addVars(K, nodes, nodes, vtype=GRB.BINARY, name="x")

    m.setObjective(
        gp.quicksum(D[i][j] * x[k, i, j] for k in range(K) for (i, j) in arcs),
        GRB.MINIMIZE
    )

    for j in C:
        m.addConstr(gp.quicksum(x[k, i, j]
                    for k in range(K) for i in nodes if i != j) == 1)
    for k in range(K):
        d_k = vehicle_depot[k]
        for h in C:
            m.addConstr(
                gp.quicksum(x[k, i, h] for i in nodes if i != h) ==
                gp.quicksum(x[k, h, j] for j in nodes if j != h)
            )
        m.addConstr(gp.quicksum(x[k, d_k, j] for j in nodes if j != d_k) <= 1)
        m.addConstr(gp.quicksum(x[k, i, d_k] for i in nodes if i != d_k) <= 1)
        # Capacity (aggregate)
        m.addConstr(
            gp.quicksum(demands_all[j] * x[k, i, j]
                        for (i, j) in arcs if j in C) <= Q
        )
        for d_other in dep:
            if d_other != d_k:
                for j in nodes:
                    if j != d_other:
                        m.addConstr(x[k, d_other, j] == 0)
                        m.addConstr(x[k, j, d_other] == 0)

    def subtour_callback(model, where):
        if where == GRB.Callback.MIPSOL:
            xval = model.cbGetSolution(x)
            for k in range(K):
                # Build adjacency for vehicle k
                edges = [(i, j) for (i, j) in arcs if xval[k, i, j] > 0.5]
                G = nx.DiGraph(edges)
                # Find strongly connected components excluding depots
                for component in nx.strongly_connected_components(G):
                    S = component - set(dep)
                    if len(S) >= 2 and not (component & set(dep)):
                        S_list = list(S)
                        model.cbLazy(
                            gp.quicksum(x[k, i, j]
                                        for i in S_list for j in S_list if i != j)
                            <= len(S_list) - 1
                        )

    print(f"\nModel built in {time.time()-t0:.1f}s  "
          f"| Variables: {m.NumVars:,}  Constraints: {m.NumConstrs:,}")

    m.optimize(subtour_callback)

    solve_time = time.time() - t0
    status = m.Status
    result = {
        'status': status, 'solve_time': solve_time, 'routes': {},
        'obj':  m.ObjVal  if status in (GRB.OPTIMAL, GRB.TIME_LIMIT) else None,
        'gap':  m.MIPGap  if status in (GRB.OPTIMAL, GRB.TIME_LIMIT) else None,
        'model': m,
    }
    if status in (GRB.OPTIMAL, GRB.TIME_LIMIT):
        for k in range(K):
            d_k    = vehicle_depot[k]
            active = {(i, j) for (i, j) in arcs if x[k, i, j].X > 0.5}
            if not active:
                continue
            route, current = [d_k], d_k
            for _ in range(N):
                nxt = next((j for (i, j) in active if i == current), None)
                if nxt is None or nxt == d_k:
                    route.append(d_k); break
                route.append(nxt); active.discard((current, nxt)); current = nxt
            result['routes'][k] = route
    return result

print("Callback solver defined.")

## 5 — Solve

In [ ]:
large = instance.n_customers > LARGE_INSTANCE_THRESHOLD

if large:
    print(f"⚠️  Instance has {instance.n_customers} customers > {LARGE_INSTANCE_THRESHOLD}.")
    print("   Using callback formulation + heuristic warm-start.")
    formulation = 'callback'
else:
    formulation = FORMULATION

print(f"\nSolving with Gurobi ({formulation.upper()}) …")
print(f"Time limit: {GUROBI_TIME_LIMIT}s  |  MIP gap target: {GUROBI_MIP_GAP:.0%}\n")

if formulation == 'MTZ':
    result = solve_vrp_mtz(instance, GUROBI_TIME_LIMIT, GUROBI_MIP_GAP)
else:
    result = solve_vrp_callback(instance, GUROBI_TIME_LIMIT, GUROBI_MIP_GAP)

## 6 — Solution report

In [ ]:
from gurobipy import GRB

STATUS_NAMES = {
    GRB.OPTIMAL:    "OPTIMAL",
    GRB.TIME_LIMIT: "TIME_LIMIT (feasible)",
    GRB.INFEASIBLE: "INFEASIBLE",
    GRB.INF_OR_UNBD: "INF_OR_UNBOUNDED",
}

status_str = STATUS_NAMES.get(result['status'], f"Status {result['status']}")
print(f"\n{'='*55}")
print(f"  GUROBI SOLUTION REPORT")
print(f"{'='*55}")
print(f"  Status          : {status_str}")
if result['obj'] is not None:
    print(f"  Objective (dist): {result['obj']:.4f}")
    print(f"  MIP gap         : {result['gap']:.2%}")
print(f"  Solve time      : {result['solve_time']:.1f}s")
print(f"  Routes found    : {len(result['routes'])}")

if result['routes']:
    demands_all = np.zeros(instance.n_nodes)
    for idx, c in enumerate(instance.customer_indices):
        demands_all[c] = instance.demands[idx]

    rows = []
    for k, route in result['routes'].items():
        customers_on_route = [n for n in route if n in instance.customer_indices]
        load = sum(demands_all[n] for n in customers_on_route)
        dist = sum(instance.dist_matrix[route[i]][route[i+1]]
                   for i in range(len(route)-1))
        rows.append({'Vehicle': k,
                     'Depot': instance.depot_indices[k // (instance.n_vehicles // instance.n_depots)],
                     'Stops': len(customers_on_route),
                     'Load': round(load, 2),
                     'Capacity': round(instance.vehicle_cap, 2),
                     'Utilisation': f"{load/instance.vehicle_cap:.1%}",
                     'Distance': round(dist, 3)})

    df = pd.DataFrame(rows)
    print(f"\n{df.to_string(index=False)}")
    print(f"\n  Total distance  : {df['Distance'].sum():.3f}")
    print(f"  Avg utilisation : {df['Load'].sum() / (len(df) * instance.vehicle_cap):.1%}")
print(f"{'='*55}")

## 7 — Plot solution routes

In [ ]:
def plot_solution(inst: VRPInstance, routes: dict, obj: float = None):
    fig, ax = plt.subplots(figsize=(11, 10))

    # Background nodes
    c_idx = inst.customer_indices
    d_idx = inst.depot_indices
    ax.scatter(inst.coords[c_idx, 0], inst.coords[c_idx, 1],
               c='lightgray', s=12, zorder=1, alpha=0.6)
    ax.scatter(inst.coords[d_idx, 0], inst.coords[d_idx, 1],
               marker='s', s=140, c='steelblue', edgecolors='navy',
               linewidth=1.5, zorder=5)
    for i in d_idx:
        ax.annotate(f"D{i}", inst.coords[i], textcoords='offset points',
                    xytext=(4, 4), fontsize=7, color='navy', fontweight='bold')

    colours = cm.tab20(np.linspace(0, 1, max(len(routes), 1)))
    for idx, (k, route) in enumerate(routes.items()):
        col = colours[idx % len(colours)]
        xs = inst.coords[route, 0]
        ys = inst.coords[route, 1]
        ax.plot(xs, ys, '-o', color=col, linewidth=1.2,
                markersize=4, alpha=0.8, zorder=3)
        # Arrow at midpoint
        for seg in range(len(route)-1):
            dx = xs[seg+1] - xs[seg]
            dy = ys[seg+1] - ys[seg]
            ax.annotate('', xy=(xs[seg]+dx*0.55, ys[seg]+dy*0.55),
                        xytext=(xs[seg]+dx*0.45, ys[seg]+dy*0.45),
                        arrowprops=dict(arrowstyle='->', color=col,
                                        lw=1.0), zorder=4)

    title = f"CVRP Solution | {len(routes)} routes"
    if obj:
        title += f" | Total dist = {obj:.2f}"
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("X"); ax.set_ylabel("Y")
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()


if result['routes']:
    plot_solution(instance, result['routes'], result['obj'])
else:
    print("No feasible solution found — nothing to plot.")

## 8 — Batch experiment: sweep over parameter space

In [ ]:
# Run a small sweep over tightness and distribution settings
# (reduce N_CUST for speed; increase for larger experiments)

SWEEP_N_CUSTOMERS = 250   # keep small for quick sweep
SWEEP_N_DEPOTS    = 3
SWEEP_TIME_LIMIT  = 60    # seconds per instance

experiments = [
    dict(distribution='uniform',   capacity_tightness='loose',          demand_dist='uniform'),
    dict(distribution='uniform',   capacity_tightness='medium',         demand_dist='normal'),
    dict(distribution='clustered', capacity_tightness='tight',          demand_dist='gamma'),
    dict(distribution='mixed',     capacity_tightness='near_infeasible', demand_dist='exponential'),
]

sweep_results = []
for i, cfg in enumerate(experiments):
    print(f"\n[{i+1}/{len(experiments)}] {cfg}")
    inst_i = generate_vrp_instance(
        n_customers          = SWEEP_N_CUSTOMERS,
        n_depots             = SWEEP_N_DEPOTS,
        n_vehicles_per_depot = 3,
        seed                 = i,
        **cfg
    )
    res_i = solve_vrp_mtz(inst_i, time_limit=SWEEP_TIME_LIMIT,
                          mip_gap=0.10, verbose=False)
    sweep_results.append({
        'distribution':  cfg['distribution'],
        'tightness':     cfg['capacity_tightness'],
        'demand_dist':   cfg['demand_dist'],
        'status':        STATUS_NAMES.get(res_i['status'], res_i['status']),
        'obj':           round(res_i['obj'],  2) if res_i['obj']  else None,
        'gap_%':         round(res_i['gap']*100, 2) if res_i['gap'] else None,
        'time_s':        round(res_i['solve_time'], 1),
        'n_routes':      len(res_i['routes']),
    })

df_sweep = pd.DataFrame(sweep_results)
print("\n" + "="*70)
print("SWEEP SUMMARY")
print("="*70)
print(df_sweep.to_string(index=False))

In [ ]:
# ── Bar chart of objective values across configurations ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels = [f"{r['distribution']}\n{r['tightness']}" for _, r in df_sweep.iterrows()]
objs   = df_sweep['obj'].fillna(0).tolist()
times  = df_sweep['time_s'].tolist()

bars = axes[0].bar(labels, objs, color=['steelblue','tomato','seagreen','gold'],
                   edgecolor='white', linewidth=1.5)
axes[0].set_title("Objective (total distance) by configuration", fontsize=11)
axes[0].set_ylabel("Total distance")
for bar, v in zip(bars, objs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{v:.0f}", ha='center', va='bottom', fontsize=9)

axes[1].bar(labels, times, color=['steelblue','tomato','seagreen','gold'],
            edgecolor='white', linewidth=1.5)
axes[1].set_title("Solve time (s) by configuration", fontsize=11)
axes[1].set_ylabel("Seconds")

plt.tight_layout()
plt.show()

## 9 — Export instance to JSON / CSV

In [ ]:
import json

def export_instance(inst: VRPInstance, base_name="vrp_instance"):
    """Save coordinates, demands, and distance matrix to disk."""
    # Coordinates + metadata CSV
    rows = []
    for i in inst.depot_indices:
        rows.append({'node': i, 'type': 'depot',
                     'x': inst.coords[i, 0], 'y': inst.coords[i, 1], 'demand': 0})
    for idx, c in enumerate(inst.customer_indices):
        rows.append({'node': c, 'type': 'customer',
                     'x': inst.coords[c, 0], 'y': inst.coords[c, 1],
                     'demand': inst.demands[idx]})
    pd.DataFrame(rows).to_csv(f"{base_name}_nodes.csv", index=False)

    # Distance matrix as numpy binary (compact)
    np.save(f"{base_name}_dist.npy", inst.dist_matrix)

    # Meta JSON
    meta = {
        'n_customers':   inst.n_customers,
        'n_depots':      inst.n_depots,
        'n_vehicles':    inst.n_vehicles,
        'vehicle_cap':   round(inst.vehicle_cap, 4),
        'distribution':  inst.distribution,
        'dist_type':     inst.dist_type,
        'tightness':     inst.tightness,
        'seed':          inst.seed,
        'total_demand':  round(float(inst.demands.sum()), 4),
    }
    with open(f"{base_name}_meta.json", 'w') as f:
        json.dump(meta, f, indent=2)

    print(f"Exported:\n  {base_name}_nodes.csv\n  {base_name}_dist.npy\n  {base_name}_meta.json")


export_instance(instance, base_name="vrp_instance")